## 🔄 Step 1: Reset and Seed Database

In [1]:
from pathlib import Path
import os, sys

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_scores import seed_score_providers
from app.db.seeders.seed_tools import seed_tool_providers
from app.db.seeders.seed_prompt_provider import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt GUID: ae329a90-086c-4256-ab51-b13cd84cf72a
Seeded SystemPrompt GUID: 55b8a901-e90a-425b-83d7-22f12dbff773
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.


## 🔍 Step 2: Fetch Prompt Generator Config

In [2]:
from sqlalchemy.orm import Session
from sqlalchemy import select
from app.db.connection import DB_PATH
from app.db.models import PromptProviderConfig
from sqlalchemy import create_engine

engine = create_engine(f"sqlite:///{DB_PATH}")
with Session(bind=engine) as session:
    prompt_gen_record = session.execute(
        select(PromptProviderConfig).where(PromptProviderConfig.name == "Basic Prompt Provider")
    ).scalar_one()
    prompt_gen_id = prompt_gen_record.id
    print(f"✅ Found Prompt Generator ID: {prompt_gen_id}")

✅ Found Prompt Generator ID: 1


## 🧱 Step 3: Instantiate Prompt Generator via Factory

In [3]:
from app.factories.prompt_provider_factory import PromptProviderFactory

generator = PromptProviderFactory.create(prompt_gen_id)
print("✅ Prompt Provider instantiated:", generator.__class__.__name__)

✅ Prompt Provider instantiated: BasicPromptProvider


## ✨ Step 4: Generate Prompt

In [4]:
prompt = generator.generate_prompt(
    agent_config={"input": "Hello, Codex."},
    system_config={"setting": "basic"},
    experiment_id="test_exp_001",
    round=1
)
print("✅ Generated Prompt:", prompt)


✅ Generated Prompt: User: Hello, Codex.
System: Response mode: basic


## 📋 Step 5: Verify Prompt Generation Log

In [5]:
from sqlalchemy import text

with engine.connect() as conn:
    rows = conn.execute(
        text("SELECT experiment_id, timestamp FROM prompt_generation_log")
    ).fetchall()
print("✅ Logged Prompt Generations:", rows)

✅ Logged Prompt Generations: [('test_exp_001', '2025-05-24T12:18:16.779662+00:00'), ('test_exp_001', '2025-05-24T12:20:00.982380+00:00'), ('test_exp_001', '2025-05-24T12:37:22.465348+00:00')]
